Search in the TCIA dataset patient define in the metadata that have done a chest CT. 
This step permit to narrow the patient search so the result patients can be analyzed with 3D Slicer so it can be descriminate better the breast region.

In [ ]:
import os
import pydicom
import numpy as np
import matplotlib.pyplot as plt

print("import libraries successfully")

import libraries successfully


In [ ]:
def extraction_ct_patient(base_folder):
    ct_patient_dict = {}
    extracted_files = 0
    
    print(f"start search in folder: {base_folder}...\n")
    
    for root, dirs, files in os.walk(base_folder):
        for file in files:
            if file.lower().endswith('.dcm'):
                percorso_completo = os.path.join(root, file)
                extracted_files += 1
                
                try:
                    ds = pydicom.dcmread(percorso_completo, stop_before_pixels=True)
                    
                    modality = ds.get('Modality', '').upper()
                    body_part = ds.get('BodyPartExamined', '').upper()
                    series_desc = ds.get('SeriesDescription', '').upper()
                    patient_id = str(ds.get('PatientID', 'ID_Sconosciuto'))
                    
                    # Filter by modality and body part
                    is_ct = (modality == 'CT')
                    chest_keyword = ['CHEST', 'THORAX', 'LUNG', 'TORACE']
                    is_chest = any(k in body_part or k in series_desc for k in chest_keyword)
                    
                    # Add to dictionary if it's a chest CT
                    if is_ct and is_chest:
                        if patient_id not in ct_patient_dict:
                            ct_patient_dict[patient_id] = []
                            print(f"Find new patient: {patient_id}")
                        
                        ct_patient_dict[patient_id].append(percorso_completo)
                        
                except Exception:
                    pass
                    
    print(f"\nfinish! Analised {extracted_files} file .dcm total.")
    return ct_patient_dict

In [ ]:
dataset_folder = "./dataset"

# Extract patient results
filtered_patients = extraction_ct_patient(dataset_folder)

if filtered_patients:
    print(f"\nFound {len(filtered_patients)} valid patients in total.")
    for patient, path_list in filtered_patients.items():
         print(f"Patient ID {patient}: {len(path_list)} slices found.")

Start search in folder: ./dataset...

Find new patient: MSB-05594
Find new patient: MSB-08214
Find new patient: MSB-04588
Find new patient: MSB-07224
Find new patient: MSB-01896
Find new patient: MSB-01799
Find new patient: MSB-00795
Find new patient: MSB-08876
Find new patient: MSB-00858
Find new patient: MSB-02054
Find new patient: MSB-09969
Find new patient: MSB-00727
Find new patient: MSB-02664
Find new patient: MSB-06267
Find new patient: MSB-02137
Find new patient: MSB-00587
Find new patient: MSB-02599
Find new patient: MSB-02101
Find new patient: MSB-05104
Find new patient: MSB-01188
Find new patient: MSB-08162

Finish! Analised 65798 file .dcm total.

Found 21 valid patients in total.
Patient ID MSB-05594: 123 slices found.
Patient ID MSB-08214: 2447 slices found.
Patient ID MSB-04588: 477 slices found.
Patient ID MSB-07224: 272 slices found.
Patient ID MSB-01896: 112 slices found.
Patient ID MSB-01799: 1273 slices found.
Patient ID MSB-00795: 1025 slices found.
Patient ID MSB-

With the extraction of this 21 patient, I open their DICOM file with Slicer 3D to have a better understending which patient can be cosider for the breast analysis.

The steps of this second analysis are:   
- Unterstand if the patient is Male or Female; 
- If their are Female, look through the sagital view (lateral view);
- Go through all the frames and search if the Region of Interest (Breast Region) could be manually segmented and have same peculiarities such as lymph nodes, metastasis and nodules.
- A plus, same patient have in their folder a Mammography (MG) where is very desciptive of the peculiarities define before.

In the end, The consider patient are 4:
- MSB-00727; modality: CHEST CT;
- MSB-01799; modality: CHEST CT, MG;
- MSB-02599; modality: CHEST CT;
- MSB-08876; modality:CHEST CT, MG.
